In [3]:
import os
import yaml
import pandas as pd

# === CONFIG ===
PROJECTS_DIR = r"C:\Users\Admin\OneDrive\Education\Master of Info - Thesis\Mobile App Data\Config Files"
OUTPUT_DIR = r"C:\GitHub\Android-Mobile-Apps"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "7.2-YML_List.csv")

# === CI PLATFORM DETECTION ===
def detect_ci_platform(file_path, yaml_text):
    text = yaml_text.lower()
    if ".github" in file_path.lower() or "github" in text:
        return "GitHub Actions"
    elif ".travis" in file_path.lower() or "travis" in text:
        return "Travis CI"
    elif "circleci" in file_path.lower() or "circleci" in text:
        return "CircleCI"
    elif "bitrise" in file_path.lower() or "bitrise" in text:
        return "Bitrise"
    else:
        return "Unknown"

# === TEST TYPE DETECTION ===
def detect_test_types(yaml_data):
    types = set()

    def search_keywords(obj):
        if isinstance(obj, dict):
            for k, v in obj.items():
                search_keywords(k)
                search_keywords(v)
        elif isinstance(obj, list):
            for item in obj:
                search_keywords(item)
        elif isinstance(obj, str):
            val = obj.lower()
            if 'connectedcheck' in val or 'instrumentation' in val or 'am instrument' in val:
                types.add('instrumentation_test')
            elif 'test' in val or './gradlew test' in val or 'unit' in val:
                types.add('unit_test')
            if 'firebase' in val:
                types.add('firebase_full')
            elif 'browserstack' in val:
                types.add('browserstack')
            elif 'emulator' in val:
                if 'manual' in val:
                    types.add('github_emulator_manual')
                elif 'gmd' in val:
                    types.add('github_gmd')
                elif 'compact' in val:
                    types.add('github_emulator_compact')
                else:
                    types.add('github_emulator_full')

    search_keywords(yaml_data)
    return types

# === PROCESS YAML FILES ===
results = []

for root, _, files in os.walk(PROJECTS_DIR):
    for file in files:
        if file.endswith(('.yml', '.yaml')):
            file_path = os.path.join(root, file)
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    raw = f.read().replace('\t', ' ')
                    ci_platform = detect_ci_platform(file_path, raw)
                    data = yaml.safe_load(raw)

                    test_types = detect_test_types(data)
                    test_type_str = ', '.join(sorted(test_types))

                    # Use folder name as project name
                    project_name = os.path.basename(root).lower()

                    results.append({
                        'project': project_name,
                        'ci_platform': ci_platform,
                        'test_type': test_type_str,
                        'unit_test': 'unit_test' in test_types,
                        'instrumentation_test': 'instrumentation_test' in test_types
                    })

            except Exception:
                results.append({
                    'project': os.path.basename(root).lower(),
                    'ci_platform': 'Error',
                    'test_type': '',
                    'unit_test': False,
                    'instrumentation_test': False
                })

# === EXPORT TO CSV ===
df = pd.DataFrame(results)
df.to_csv(OUTPUT_CSV, index=False)
print(f"✅ File-level YML list written to: {OUTPUT_CSV}")


✅ File-level YML list written to: C:\GitHub\Android-Mobile-Apps\7.2-YML_List.csv
